# 04 Drive

Runs the driving program (`drive.py`): camera -> perception -> decision -> control -> motors.
Every run is logged to `/workspace/usb/logs/`.

1. Put the robot on the lane and check the battery (at least 11.4 V at rest).
2. Choose the mode and the duration, click **start**.
3. **stop** ends the run at once. From a terminal: `jetbot/scripts/stop_robot.sh`.

In [ ]:
import threading
import ipywidgets.widgets as widgets
from IPython.display import display

from drive import run

mode = widgets.Dropdown(description='mode', options=['adaptive', 'baseline', 'nocurve'], value='adaptive')
seconds = widgets.FloatText(description='seconds', value=30.0)
start_button = widgets.Button(description='start', button_style='success')
stop_button = widgets.Button(description='stop', button_style='danger')
status = widgets.Label(value='ready')
stop_event = threading.Event()


def drive_in_background():
    reason, log_dir = run(mode.value, seconds.value, baseline=mode.value == 'baseline',
                          use_curve=mode.value == 'adaptive', stop_event=stop_event)
    status.value = '%s, log in %s' % (reason, log_dir)


def start(button):
    stop_event.clear()
    status.value = 'driving (%s)' % mode.value
    threading.Thread(target=drive_in_background).start()


start_button.on_click(start)
stop_button.on_click(lambda button: stop_event.set())
display(widgets.VBox([mode, seconds, widgets.HBox([start_button, stop_button]), status]))